In [81]:
class Vector:
    def __init__(self, x=0, y=0):
        self.x, self.y = x, y
    
    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)
    
    def __sub__(self, other):
        return Vector(self.x - other.x, self.y - other.y)
    
    def __mul__(self, scalar):
        return Vector(self.x * scalar, self.y * scalar)
    
    def __truediv__(self, scalar):
        return Vector(self.x / scalar, self.y / scalar)
    
    def copy(self):
        return Vector(self.x, self.y)

class MockKeyframe:
    def __init__(self, co, handle_right=None, handle_left=None):
        self.co = co
        self.handle_right = handle_right or co.copy()
        self.handle_left = handle_left or co.copy()

class _FnBezier:
    def __init__(self, p0, p1, p2, p3):
        self._p0, self._p1, self._p2, self._p3 = p0, p1, p2, p3
    
    @property
    def points(self):
        return self._p0, self._p1, self._p2, self._p3
    
    @classmethod
    def from_fcurve(cls, kp0, kp1):
        p0, p1, p2, p3 = kp0.co, kp0.handle_right, kp1.handle_left, kp1.co
        return cls(p0, p1, p2, p3)

def setInterpolation(bezier_params, kp0_co, kp1_co):
    is_linear = bezier_params[0] == bezier_params[1] and bezier_params[2] == bezier_params[3]
    if is_linear:
        return "LINEAR", kp0_co, kp1_co
    
    d = (kp1_co - kp0_co) / 127.0
    handle_right = kp0_co + Vector(d.x * bezier_params[0], d.y * bezier_params[1])
    handle_left = kp0_co + Vector(d.x * bezier_params[2], d.y * bezier_params[3])
    
    return "BEZIER", handle_right, handle_left

def toVMDControlPoints_original(bezier):
    p0, p1, p2, p3 = bezier.points
    dx = p3.x - p0.x
    dy = p3.y - p0.y
    
    # 计算相对位置
    x1, y1 = p1.x - p0.x, p1.y - p0.y
    x2, y2 = p2.x - p0.x, p2.y - p0.y
    
    # 处理 X 坐标
    print(x1,x2)
    print(dx)
    if abs(dx) < 1e-6:
        x1, x2 = 20, 107
        x1 = max(0, min(127, int(0.5 + x1 * 127.0 / dx)))
        x2 = max(0, min(127, int(0.5 + x2 * 127.0 / dx)))
    else:
        x1 = max(0, min(127, int(0.5 + x1 * 127.0 / dx)))
        x2 = max(0, min(127, int(0.5 + x2 * 127.0 / dx)))
    
    # 处理 Y 坐标
    if abs(dy) < 1e-6:
        y1, y2 = 20, 107
    else:
        y1 = max(0, min(127, int(0.5 + y1 * 127.0 / dy)))
        y2 = max(0, min(127, int(0.5 + y2 * 127.0 / dy)))
    
    # 返回格式选择
    return_as_tuples = False  # 设置为 True 返回元组格式，False 返回列表格式
    
    if return_as_tuples:
        return ((x1, y1), (x2, y2))
    else:
        return [x1, y1, x2, y2]

def test_vmd_roundtrip(original_params, kp0, kp1):
    kp0_vec = Vector(kp0[0], kp0[1])
    kp1_vec = Vector(kp1[0], kp1[1])
    
    interp_type, handle_right, handle_left = setInterpolation(original_params, kp0_vec, kp1_vec)
    
    if interp_type == "LINEAR":
        return original_params, [20, 20, 107, 107]
    
    kp0_obj = MockKeyframe(kp0_vec, handle_right, handle_right)
    kp1_obj = MockKeyframe(kp1_vec, handle_left, handle_left)
    
    bezier = _FnBezier.from_fcurve(kp0_obj, kp1_obj)
    result_params = toVMDControlPoints_original(bezier)
    
    return original_params, result_params

def main():
    # 🔧 在这里修改你要测试的参数
    input_params = [99, 0, 73, 127]
    kp0 = (4712, 0.9801)
    kp1 = (4718, 0.9801)
    
    original, result = test_vmd_roundtrip(input_params, kp0, kp1)
    
    print(f"输入: {original}")
    print(f"输出: {result}")

    input_params = [61, 0, 107, 107]
    kp0 = (1510.8818, 1.0000)
    kp1 = (1513.0552, 1.0000)
    
    original, result = test_vmd_roundtrip(input_params, kp0, kp1)
    
    print(f"输入: {original}")
    print(f"输出: {result}")

    input_params = [78, 0, 57, 127]
    kp0 = (1510.8818, 1.0000)
    kp1 = (1513.0552, 1.0000)
    
    original, result = test_vmd_roundtrip(input_params, kp0, kp1)
    
    print(f"输入: {original}")
    print(f"输出: {result}")
    
    input_params = [20, 20, 76, 127]
    kp0 = (2318.6299, 0.9999)
    kp1 = (2320.3938, 0.9999)
    
    original, result = test_vmd_roundtrip(input_params, kp0, kp1)
    
    print(f"输入: {original}")
    print(f"输出: {result}")
    
    input_params = [20, 20, 68, 127]
    kp0 = (3230.1023, 0.9999)
    kp1 = (3232.7480, 0.9999)
    
    original, result = test_vmd_roundtrip(input_params, kp0, kp1)
    
    print(f"输入: {original}")
    print(f"输出: {result}")
    
    input_params = [20, 40, 87, 87]
    kp0 = (326.3150, 0.9884)
    kp1 = (327.6850, 0.9479)
    
    original, result = test_vmd_roundtrip(input_params, kp0, kp1)
    
    print(f"输入: {original}")
    print(f"输出: {result}")
    
if __name__ == "__main__":
    main()

4.677165354331009 3.448818897637466
6
输入: [99, 0, 73, 127]
输出: [99, 20, 73, 107]
1.043916535433027 1.8311322834645125
2.1733999999999014
输入: [61, 0, 107, 107]
输出: [61, 20, 107, 107]
1.3348440944880622 0.9754629921260403
2.1733999999999014
输入: [78, 0, 57, 127]
输出: [78, 20, 57, 107]
0.2777795275592325 1.0555622047245379
1.7638999999999214
输入: [20, 20, 76, 127]
输出: [20, 20, 76, 107]
0.4166456692914835 1.41659527559068
2.6457000000000335
输入: [20, 20, 68, 127]
输出: [20, 20, 68, 107]
0.21574803149604804 0.9385039370078516
1.3700000000000045
输入: [20, 40, 87, 87]
输出: [20, 40, 87, 87]


In [69]:
#!/usr/bin/env python3
"""
VMD插值参数往返转换随机测试脚本 - 10000个测试用例
"""
import random

class Vector:
    def __init__(self, x=0, y=0):
        self.x, self.y = x, y
    
    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)
    
    def __sub__(self, other):
        return Vector(self.x - other.x, self.y - other.y)
    
    def __mul__(self, scalar):
        return Vector(self.x * scalar, self.y * scalar)
    
    def __truediv__(self, scalar):
        return Vector(self.x / scalar, self.y / scalar)
    
    def copy(self):
        return Vector(self.x, self.y)

class MockKeyframe:
    def __init__(self, co, handle_right=None, handle_left=None):
        self.co = co
        self.handle_right = handle_right or co.copy()
        self.handle_left = handle_left or co.copy()

class _FnBezier:
    def __init__(self, p0, p1, p2, p3):
        self._p0, self._p1, self._p2, self._p3 = p0, p1, p2, p3
    
    @property
    def points(self):
        return self._p0, self._p1, self._p2, self._p3
    
    @classmethod
    def from_fcurve(cls, kp0, kp1):
        p0, p1, p2, p3 = kp0.co, kp0.handle_right, kp1.handle_left, kp1.co
        return cls(p0, p1, p2, p3)

def setInterpolation(bezier_params, kp0_co, kp1_co):
    is_linear = bezier_params[0] == bezier_params[1] and bezier_params[2] == bezier_params[3]
    if is_linear:
        return "LINEAR", kp0_co, kp1_co
    
    d = (kp1_co - kp0_co) / 127.0
    handle_right = kp0_co + Vector(d.x * bezier_params[0], d.y * bezier_params[1])
    handle_left = kp0_co + Vector(d.x * bezier_params[2], d.y * bezier_params[3])
    
    return "BEZIER", handle_right, handle_left

def toVMDControlPoints_original(bezier):
    p0, p1, p2, p3 = bezier.points
    dx = p3.x - p0.x
    dy = p3.y - p0.y
    
    # 计算相对位置
    x1, y1 = p1.x - p0.x, p1.y - p0.y
    x2, y2 = p2.x - p0.x, p2.y - p0.y
    
    # 处理 X 坐标
    if abs(dx) < 1e-6:
        x1, x2 = 0, 127
    else:
        x1 = max(0, min(127, int(0.5 + x1 * 127.0 / dx)))
        x2 = max(0, min(127, int(0.5 + x2 * 127.0 / dx)))
    
    # 处理 Y 坐标
    if abs(dy) < 1e-6:
        y1, y2 = 0, 127
    else:
        y1 = max(0, min(127, int(0.5 + y1 * 127.0 / dy)))
        y2 = max(0, min(127, int(0.5 + y2 * 127.0 / dy)))
    
    return [x1, y1, x2, y2]

def test_vmd_roundtrip(original_params, kp0, kp1):
    kp0_vec = Vector(kp0[0], kp0[1])
    kp1_vec = Vector(kp1[0], kp1[1])
    
    interp_type, handle_right, handle_left = setInterpolation(original_params, kp0_vec, kp1_vec)
    
    if interp_type == "LINEAR":
        return original_params, [20, 20, 107, 107]
    
    kp0_obj = MockKeyframe(kp0_vec, handle_right, handle_right)
    kp1_obj = MockKeyframe(kp1_vec, handle_left, handle_left)
    
    bezier = _FnBezier.from_fcurve(kp0_obj, kp1_obj)
    result_params = toVMDControlPoints_original(bezier)
    
    return original_params, result_params

def generate_random_test_case():
    """生成随机测试用例，避免生成线性插值的情况（a==b且c==d）"""
    while True:
        # input_params: 0~127的整数
        input_params = [random.randint(0, 127) for _ in range(4)]
        
        # 检查是否为线性插值情况（a==b且c==d）
        if input_params[0] == input_params[1] and input_params[2] == input_params[3]:
            continue  # 重新生成
        
        # kp0: (0.0~10.0, -1.0~1.0)
        kp0 = (
            random.uniform(0.0, 10.0),
            random.uniform(-1.0, 1.0)
        )
        
        # kp1: (0.0~10.0, -1.0~1.0)
        kp1 = (
            random.uniform(0.0, 10.0),
            random.uniform(-1.0, 1.0)
        )
        
        # 确保kp1.x > kp0.x (时间递增)
        if kp1[0] <= kp0[0]:
            kp1 = (kp0[0] + random.uniform(0.1, 2.0), kp1[1])
        
        return input_params, kp0, kp1

def calculate_error(original, result):
    """计算误差"""
    if len(original) != len(result):
        return float('inf')
    
    total_error = 0
    for i in range(len(original)):
        total_error += abs(original[i] - result[i])
    
    return total_error

def main():
    total_tests = 2000000
    error_count = 0
    perfect_count = 0
    
    # 统计信息
    max_error = 0
    error_distribution = {0: 0, 1: 0, 2: 0, 5: 0, 10: 0, 20: 0, 'large': 0}
    
    for test_num in range(total_tests):
        # 生成随机测试用例
        input_params, kp0, kp1 = generate_random_test_case()
        
        try:
            # 执行往返转换测试
            original, result = test_vmd_roundtrip(input_params, kp0, kp1)
            
            # 计算误差
            error = calculate_error(original, result)
            
            # 更新统计
            if error == 0:
                perfect_count += 1
            else:
                error_count += 1
                
                # 只有出现误差时才打印
                print(f"测试 #{test_num + 1} - 发现误差:")
                print(f"  输入参数: {original}")
                print(f"  kp0: {kp0}")
                print(f"  kp1: {kp1}")
                print(f"  输出结果: {result}")
                print(f"  误差: {error}")
                print("-" * 40)
            
            # 统计误差分布
            max_error = max(max_error, error)
            if error == 0:
                error_distribution[0] += 1
            elif error <= 1:
                error_distribution[1] += 1
            elif error <= 2:
                error_distribution[2] += 1
            elif error <= 5:
                error_distribution[5] += 1
            elif error <= 10:
                error_distribution[10] += 1
            elif error <= 20:
                error_distribution[20] += 1
            else:
                error_distribution['large'] += 1
        
        except Exception as e:
            print(f"测试 #{test_num + 1} - 出现异常:")
            print(f"  输入参数: {input_params}")
            print(f"  kp0: {kp0}")
            print(f"  kp1: {kp1}")
            print(f"  异常信息: {e}")
            print("-" * 40)
            error_count += 1
    
    # 打印总结报告
    print("\n" + "=" * 60)
    print("测试完成 - 总结报告")
    print("=" * 60)
    print(f"总测试数量: {total_tests}")
    print(f"完美匹配: {perfect_count} ({(perfect_count/total_tests)*100:.2f}%)")
    print(f"有误差的: {error_count} ({(error_count/total_tests)*100:.2f}%)")
    print(f"最大误差: {max_error}")
    
    print("\n误差分布:")
    print(f"  误差 = 0:     {error_distribution[0]:5d} ({(error_distribution[0]/total_tests)*100:.2f}%)")
    print(f"  误差 ≤ 1:     {error_distribution[1]:5d} ({(error_distribution[1]/total_tests)*100:.2f}%)")
    print(f"  误差 ≤ 2:     {error_distribution[2]:5d} ({(error_distribution[2]/total_tests)*100:.2f}%)")
    print(f"  误差 ≤ 5:     {error_distribution[5]:5d} ({(error_distribution[5]/total_tests)*100:.2f}%)")
    print(f"  误差 ≤ 10:    {error_distribution[10]:5d} ({(error_distribution[10]/total_tests)*100:.2f}%)")
    print(f"  误差 ≤ 20:    {error_distribution[20]:5d} ({(error_distribution[20]/total_tests)*100:.2f}%)")
    print(f"  误差 > 20:    {error_distribution['large']:5d} ({(error_distribution['large']/total_tests)*100:.2f}%)")

if __name__ == "__main__":
    main()

测试 #1674448 - 发现误差:
  输入参数: [12, 13, 109, 60]
  kp0: (9.235692253975353, 0.7431406625859833)
  kp1: (9.235692481865485, 0.4080043080473197)
  输出结果: [0, 13, 127, 60]
  误差: 30
----------------------------------------

测试完成 - 总结报告
总测试数量: 2000000
完美匹配: 1999999 (100.00%)
有误差的: 1 (0.00%)
最大误差: 30

误差分布:
  误差 = 0:     1999999 (100.00%)
  误差 ≤ 1:         0 (0.00%)
  误差 ≤ 2:         0 (0.00%)
  误差 ≤ 5:         0 (0.00%)
  误差 ≤ 10:        0 (0.00%)
  误差 ≤ 20:        0 (0.00%)
  误差 > 20:        1 (0.00%)
